# 4 — Coğrafi Veri İşleme

Bu notebook, ilan verisindeki **il / ilçe** konumlarını coğrafi koordinata
(lat/lon) çevirir ve her ilçe için **il merkezine uzaklık (km)** üretir. Bu
öznitelikler modele coğrafi sinyal ekler (target encoding'in ötesinde mutlak
konum: doğu/batı, kıyı/iç bölge vb.).

**Neden ilçe düzeyi?** Veri setinde ~4600 benzersiz *mahalle* var; her birini
Nominatim ile geocode etmek (1 istek/sn) çok uzun sürer ve engellenir. ~460
*ilçe* ise makuldür ve fiyat için yeterli coğrafi çözünürlük sağlar.

**Asıl iş `src/cografi_zenginlestirme.py` scriptinde** (resumable, kademeli kayıt).
Bu notebook akışı açıklar ve sonucu inceler. Eski sürüm kaldırılan
`ox.geometries...` API'sini kullanıyordu; osmnx 2.x için yeniden yazıldı.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")
from model_egitimi import temizle  # ham veriyi temizleyen ortak fonksiyon

## 1. Benzersiz konumları çıkar

In [ ]:
df = temizle(pd.read_csv("../data/raw/turkey_rent.csv", low_memory=False))
print("Benzersiz il        :", df["il"].nunique())
print("Benzersiz il+ilçe   :", df.groupby(["il", "ilce"]).ngroups)
print("Benzersiz il+ilçe+mh:", df.groupby(["il", "ilce", "mahalle"]).ngroups)

## 2. İlçe geocoding

Geocoding işlemi uzun sürdüğü için terminalden çalıştırılır:

```bash
python src/cografi_zenginlestirme.py
```

Script resumable'dır: yarıda kesilse bile tekrar çalıştırınca kaldığı yerden
devam eder. Çıktı: `data/geo/ilce_latlon.csv`. Aşağıda hazır sonucu yüklüyoruz.

In [ ]:
geo = pd.read_csv("../data/geo/ilce_latlon.csv")
print("Geocode edilmiş ilçe:", len(geo))
geo.head()

## 3. İl merkezine uzaklık

Her il için merkez, o ile ait ilçelerin koordinat ortalamasıdır; uzaklık
haversine ile km cinsinden hesaplanır (script tarafından eklenir).

In [ ]:
geo["il_merkeze_uzaklik_km"].describe().round(1)

## 4. Türkiye haritası (ilçe koordinatları)

Lat/lon dağılımı kabaca Türkiye haritasını vermeli — geocoding'in doğruluğunu
görsel olarak kontrol eder.

In [ ]:
plt.figure(figsize=(11, 5))
plt.scatter(geo["lon"], geo["lat"], s=12, alpha=0.6)
plt.xlabel("Boylam (lon)")
plt.ylabel("Enlem (lat)")
plt.title("Geocode edilmiş ilçeler")
plt.grid(alpha=0.3)
plt.show()

## 5. Modele entegrasyon

Bu tablo `src/model_egitimi.py` içindeki `temizle()` tarafından otomatik olarak
merge edilir; modele `ilce_lat`, `ilce_lon`, `il_merkeze_uzaklik_km` öznitelikleri
olarak girer. İlçe bulunamazsa il merkezine, o da yoksa global ortalamaya düşülür.

Coğrafi öznitelikler eklenince test R² **0.70 → 0.71**, CV R² **0.70 → 0.72**
seviyesine çıktı.

### Sonraki adımlar
- Mahalle düzeyi koordinat (toplu/önbellekli geocoding ile) daha ince sinyal verir.
- Şehir merkezine / sahile / metro hattına uzaklık gibi POI temelli öznitelikler.